# 02. Feature Engineering

Este notebook realiza la preparación de datos y la ingeniería de variables para el conjunto de datos del proceso de flotación de la planta minera.

## Objetivos:
1. **Limpieza y preprocesamiento de datos:**
   - Convertir la columna `date` a tipo `datetime`.
   - Reemplazar comas (`,`) por puntos (`.`) en variables numéricas y convertirlas a tipo `float`.
2. **Ingeniería de variables (Ventanas Móviles):**
   - Calcular la media móvil y la desviación estándar móvil para las ventanas de **3, 6 y 12 observaciones** de las variables de proceso controlables.
3. **División cronológica (Train/Test Split):**
   - Dividir el conjunto de datos en 80% entrenamiento y 20% prueba cronológicamente, previniendo fuga de datos temporales (data leakage).
4. **Almacenamiento de datos procesados:**
   - Guardar los datos finales en formato Parquet en `data/processed/` para mejorar la velocidad y eficiencia en el modelado.

In [1]:
import pandas as pd
import numpy as np
import os

print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)

Pandas version: 3.0.2
Numpy version: 2.2.6


In [2]:
raw_data_path = '../data/raw/MiningProcess_Flotation_Plant_Database.csv'
print(f"Cargando datos desde {raw_data_path}...")
df = pd.read_csv(raw_data_path)
print(f"Dimensiones originales: {df.shape}")

Cargando datos desde ../data/raw/MiningProcess_Flotation_Plant_Database.csv...
Dimensiones originales: (737453, 24)


In [3]:
print("Limpiando datos...")

# Convertir la columna date a datetime
df['date'] = pd.to_datetime(df['date'])

# Reemplazar comas por puntos en todas las columnas excepto 'date' y convertirlas a float
for col in df.columns:
    if col != 'date':
        df[col] = df[col].astype(str).str.replace(',', '.').astype(float)

# Verificar tipos de datos
print("\nTipos de datos actualizados:")
print(df.dtypes)
print("\nPrimeras filas procesadas:")
df.head()

Limpiando datos...

Tipos de datos actualizados:
date                            datetime64[us]
% Iron Feed                            float64
% Silica Feed                          float64
Starch Flow                            float64
Amina Flow                             float64
Ore Pulp Flow                          float64
Ore Pulp pH                            float64
Ore Pulp Density                       float64
Flotation Column 01 Air Flow           float64
Flotation Column 02 Air Flow           float64
Flotation Column 03 Air Flow           float64
Flotation Column 04 Air Flow           float64
Flotation Column 05 Air Flow           float64
Flotation Column 06 Air Flow           float64
Flotation Column 07 Air Flow           float64
Flotation Column 01 Level              float64
Flotation Column 02 Level              float64
Flotation Column 03 Level              float64
Flotation Column 04 Level              float64
Flotation Column 05 Level              float64
Flotation C

,date,% Iron Feed,% Silica Feed,Starch Flow,Amina Flow,Ore Pulp Flow,Ore Pulp pH,Ore Pulp Density,Flotation Column 01 Air Flow,Flotation Column 02 Air Flow,...,Flotation Column 07 Air Flow,Flotation Column 01 Level,Flotation Column 02 Level,Flotation Column 03 Level,Flotation Column 04 Level,Flotation Column 05 Level,Flotation Column 06 Level,Flotation Column 07 Level,% Iron Concentrate,% Silica Concentrate
0,2017-03-10 01:00:00,55.2,16.98,3019.53,557.434,395.713,10.0664,1.74,249.214,253.235,...,250.884,457.396,432.962,424.954,443.558,502.255,446.370,523.344,66.91,1.31
1,2017-03-10 01:00:00,55.2,16.98,3024.41,563.965,397.383,10.0672,1.74,249.719,250.532,...,248.994,451.891,429.560,432.939,448.086,496.363,445.922,498.075,66.91,1.31
2,2017-03-10 01:00:00,55.2,16.98,3043.46,568.054,399.668,10.0680,1.74,249.741,247.874,...,248.071,451.240,468.927,434.610,449.688,484.411,447.826,458.567,66.91,1.31
3,2017-03-10 01:00:00,55.2,16.98,3047.36,568.665,397.939,10.0689,1.74,249.917,254.487,...,251.147,452.441,458.165,442.865,446.210,471.411,437.690,427.669,66.91,1.31
4,2017-03-10 01:00:00,55.2,16.98,3033.69,558.167,400.254,10.0697,1.74,250.203,252.136,...,248.928,452.441,452.900,450.523,453.670,462.598,443.682,425.679,66.91,1.31


In [4]:
# Identificar las variables de proceso controlables
controllable_cols = [
    'Amina Flow',
    'Ore Pulp pH',
    'Flotation Column 01 Air Flow',
    'Flotation Column 02 Air Flow',
    'Flotation Column 03 Air Flow',
    'Flotation Column 04 Air Flow',
    'Flotation Column 05 Air Flow',
    'Flotation Column 06 Air Flow',
    'Flotation Column 07 Air Flow',
    'Flotation Column 01 Level',
    'Flotation Column 02 Level',
    'Flotation Column 03 Level',
    'Flotation Column 04 Level',
    'Flotation Column 05 Level',
    'Flotation Column 06 Level',
    'Flotation Column 07 Level'
]

print("Generando variables con ventanas móviles (mean y std de 3, 6, y 12 observaciones)...")

windows = [3, 6, 12]
new_features = {}

# Creamos un diccionario con las nuevas características para concatenarlas de golpe
# y evitar advertencias sobre DataFrames fragmentados
for col in controllable_cols:
    for w in windows:
        # Calcular media móvil
        new_features[f'{col}_roll_mean_{w}'] = df[col].rolling(window=w).mean()
        # Calcular desviación estándar móvil
        new_features[f'{col}_roll_std_{w}'] = df[col].rolling(window=w).std()

# Crear un DataFrame con las nuevas características e indexar igual que el original
df_new_features = pd.DataFrame(new_features, index=df.index)

# Concatenar
df = pd.concat([df, df_new_features], axis=1)

print(f"Dimensiones después de agregar características: {df.shape}")

# Eliminar las filas iniciales que tengan valores nulos debidos a las ventanas móviles
print("\nFilas nulas antes de dropna:", df.isna().any(axis=1).sum())
df = df.dropna().reset_index(drop=True)
print("Dimensiones después de dropna:", df.shape)

Generando variables con ventanas móviles (mean y std de 3, 6, y 12 observaciones)...
Dimensiones después de agregar características: (737453, 120)

Filas nulas antes de dropna: 11
Dimensiones después de dropna: (737442, 120)


In [5]:
print("Realizando la división cronológica en Train/Test (80/20)...")

# Ordenar por fecha por seguridad
df = df.sort_values(by='date').reset_index(drop=True)

# Calcular índice de split aproximado
split_ratio = 0.8
split_idx = int(len(df) * split_ratio)

# Determinar la fecha exacta del split para evitar que el mismo timestamp horario
# quede dividido entre entrenamiento y prueba
split_date = df['date'].iloc[split_idx]
print(f"Fecha límite de corte seleccionada: {split_date}")

# Split cronológico
train_df = df[df['date'] < split_date].copy()
test_df = df[df['date'] >= split_date].copy()

# Validaciones de consistencia
print(f"\nConjunto de entrenamiento: {train_df.shape} (desde {train_df['date'].min()} hasta {train_df['date'].max()})")
print(f"Conjunto de prueba: {test_df.shape} (desde {test_df['date'].min()} hasta {test_df['date'].max()})")
print(f"Proporción real: Train {len(train_df)/len(df):.1%}, Test {len(test_df)/len(df):.1%}")

# Validar que no haya fugas (Overlap)
train_dates = set(train_df['date'].unique())
test_dates = set(test_df['date'].unique())
overlap = train_dates.intersection(test_dates)

print(f"\nCantidad de marcas de tiempo solapadas: {len(overlap)}")
assert len(overlap) == 0, "¡Error de Leakage! Hay marcas de tiempo compartidas entre train y test."
print("¡División exitosa sin solapamiento temporal!")

Realizando la división cronológica en Train/Test (80/20)...
Fecha límite de corte seleccionada: 2017-08-06 20:00:00

Conjunto de entrenamiento: (589842, 120) (desde 2017-03-10 01:00:00 hasta 2017-08-06 19:00:00)
Conjunto de prueba: (147600, 120) (desde 2017-08-06 20:00:00 hasta 2017-09-09 23:00:00)
Proporción real: Train 80.0%, Test 20.0%

Cantidad de marcas de tiempo solapadas: 0
¡División exitosa sin solapamiento temporal!


In [6]:
# Crear la ruta de salida si no existe
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

train_output_path = os.path.join(output_dir, 'train_engineered.parquet')
test_output_path = os.path.join(output_dir, 'test_engineered.parquet')

print(f"Guardando conjunto de entrenamiento en: {train_output_path}...")
train_df.to_parquet(train_output_path, index=False)

print(f"Guardando conjunto de prueba en: {test_output_path}...")
test_df.to_parquet(test_output_path, index=False)

print("\n¡Archivos guardados con éxito en data/processed/!")

Guardando conjunto de entrenamiento en: ../data/processed\train_engineered.parquet...
Guardando conjunto de prueba en: ../data/processed\test_engineered.parquet...

¡Archivos guardados con éxito en data/processed/!


In [7]:
print("Verificando integridad de los archivos guardados...")

train_output_path = '../data/processed/train_engineered.parquet'
test_output_path = '../data/processed/test_engineered.parquet'

train_loaded = pd.read_parquet(train_output_path)
test_loaded = pd.read_parquet(test_output_path)

print(f"\nDatos de entrenamiento cargados correctamente: {train_loaded.shape}")
print(f"Datos de prueba cargados correctamente: {test_loaded.shape}")

# Validar que la variable objetivo '% Silica Concentrate' no tenga nulos
print(f"\nNulos en la variable objetivo en Train: {train_loaded['% Silica Concentrate'].isna().sum()}")
print(f"Nulos en la variable objetivo en Test: {test_loaded['% Silica Concentrate'].isna().sum()}")

print("\n¡Verificación final exitosa! El pipeline de Feature Engineering funciona correctamente.")

Verificando integridad de los archivos guardados...

Datos de entrenamiento cargados correctamente: (589842, 120)
Datos de prueba cargados correctamente: (147600, 120)

Nulos en la variable objetivo en Train: 0
Nulos en la variable objetivo en Test: 0

¡Verificación final exitosa! El pipeline de Feature Engineering funciona correctamente.
